# Phase 3 — XOR, échec du neurone unique, victoire de la couche cachée

## Objectif

Montrer pourquoi un neurone unique ne peut pas résoudre XOR, puis utiliser un réseau 2–2–1 pour le résoudre.

Dans cette phase, on va :
- définir le dataset XOR ;
- entraîner un réseau 2–2–1 en numpy ;
- tester plusieurs initialisations ;
- garder le meilleur modèle ;
- exporter la courbe de loss ;
- exporter la frontière de décision.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [26]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_xor = np.array([0, 1, 1, 0], dtype=float)

print("X_xor =\n", X_xor)
print("y_xor =", y_xor)

X_xor =
 [[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
y_xor = [0. 1. 1. 0.]


In [27]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def tanh_grad(a):
    return 1 - a**2

def compute_loss_bce(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

## 1. Idée

Un seul entraînement peut tomber dans une mauvaise solution.

On va donc :
- entraîner plusieurs réseaux 2–2–1 avec des seeds différentes ;
- garder celui qui obtient la meilleure accuracy ;
- tracer les résultats du meilleur modèle.

In [28]:
def train_xor(seed, learning_rate=0.1, n_epochs=20000):
    np.random.seed(seed)

    W1 = np.random.randn(2, 2) * 0.5
    b1 = np.zeros((1, 2))

    W2 = np.random.randn(2, 1) * 0.5
    b2 = np.zeros((1, 1))

    losses = []

    for epoch in range(n_epochs):
        # Forward pass
        z1 = np.dot(X_xor, W1) + b1
        a1 = tanh(z1)

        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)

        y_pred = a2.flatten()

        # Loss
        loss = compute_loss_bce(y_xor, y_pred)
        losses.append(loss)

        # Backprop sortie
        error2 = (y_pred - y_xor).reshape(-1, 1)
        dW2 = (1 / len(X_xor)) * np.dot(a1.T, error2)
        db2 = (1 / len(X_xor)) * np.sum(error2, axis=0, keepdims=True)

        # Backprop cachée
        error1 = np.dot(error2, W2.T) * tanh_grad(a1)
        dW1 = (1 / len(X_xor)) * np.dot(X_xor.T, error1)
        db1 = (1 / len(X_xor)) * np.sum(error1, axis=0, keepdims=True)

        # Update
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    final_pred = y_pred
    final_acc = np.mean((final_pred > 0.5) == y_xor)
    final_loss = losses[-1]

    return {
        "seed": seed,
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
        "losses": losses,
        "y_pred": final_pred,
        "acc": final_acc,
        "loss": final_loss
    }

In [29]:
results = []

for seed in range(20):
    result = train_xor(seed=seed, learning_rate=0.1, n_epochs=20000)
    results.append(result)
    print(f"Seed {seed:2d} | Loss: {result['loss']:.4f} | Accuracy: {result['acc']:.2%}")

Seed  0 | Loss: 0.3473 | Accuracy: 50.00%
Seed  1 | Loss: 0.3473 | Accuracy: 50.00%
Seed  2 | Loss: 0.3473 | Accuracy: 50.00%
Seed  3 | Loss: 0.0016 | Accuracy: 100.00%
Seed  4 | Loss: 0.0016 | Accuracy: 100.00%
Seed  5 | Loss: 0.0016 | Accuracy: 100.00%
Seed  6 | Loss: 0.0016 | Accuracy: 100.00%
Seed  7 | Loss: 0.0016 | Accuracy: 100.00%
Seed  8 | Loss: 0.3473 | Accuracy: 50.00%
Seed  9 | Loss: 0.0017 | Accuracy: 100.00%
Seed 10 | Loss: 0.0016 | Accuracy: 100.00%
Seed 11 | Loss: 0.3473 | Accuracy: 50.00%
Seed 12 | Loss: 0.0016 | Accuracy: 100.00%
Seed 13 | Loss: 0.3473 | Accuracy: 50.00%
Seed 14 | Loss: 0.0016 | Accuracy: 100.00%
Seed 15 | Loss: 0.6931 | Accuracy: 50.00%
Seed 16 | Loss: 0.3473 | Accuracy: 50.00%
Seed 17 | Loss: 0.3473 | Accuracy: 50.00%
Seed 18 | Loss: 0.0016 | Accuracy: 100.00%
Seed 19 | Loss: 0.3473 | Accuracy: 50.00%


In [30]:
best_result = sorted(results, key=lambda r: (-r["acc"], r["loss"]))[0]

W1 = best_result["W1"]
b1 = best_result["b1"]
W2 = best_result["W2"]
b2 = best_result["b2"]
losses = best_result["losses"]
y_pred_final = best_result["y_pred"]
acc_final = best_result["acc"]

print("\nMeilleur modèle")
print("Seed retenu      :", best_result["seed"])
print("Loss finale      :", round(best_result["loss"], 4))
print("Accuracy finale  :", f"{acc_final:.2%}")
print("Prédictions      :", y_pred_final.round(3))
print("Classes prédites :", (y_pred_final > 0.5).astype(int))
print("Vraies classes   :", y_xor.astype(int))


Meilleur modèle
Seed retenu      : 10
Loss finale      : 0.0016
Accuracy finale  : 100.00%
Prédictions      : [0.001 0.998 0.998 0.001]
Classes prédites : [0 1 1 0]
Vraies classes   : [0 1 1 0]


In [32]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss BCE")
plt.title("Convergence du meilleur réseau 2–2–1 sur XOR")
plt.grid(True)
plt.savefig("phase3_loss_curve.png", dpi=100, bbox_inches="tight")
plt.show()

print("Courbe sauvegardée : phase3_loss_curve.png")

Courbe sauvegardée : phase3_loss_curve.png


C:\Users\serge\AppData\Local\Temp\ipykernel_25392\3898141602.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
xx, yy = np.meshgrid(
    np.linspace(-0.5, 1.5, 200),
    np.linspace(-0.5, 1.5, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]

z1g = tanh(np.dot(grid, W1) + b1)
z2g = sigmoid(np.dot(z1g, W2) + b2).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, z2g, alpha=0.4, cmap="RdBu")
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, s=100, cmap="RdBu", edgecolors="k")
plt.title("XOR : frontière de décision du meilleur réseau 2-2-1")
plt.xlabel("x1")
plt.ylabel("x2")
plt.savefig("phase3_xor_boundary.png", dpi=100, bbox_inches="tight")
plt.show()

print("Frontière sauvegardée : phase3_xor_boundary.png")

Frontière sauvegardée : phase3_xor_boundary.png


C:\Users\serge\AppData\Local\Temp\ipykernel_25392\613769148.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
